### 6.2 Distribution of Machine Types

In [ ]:
df[["Type_L", "Type_M"]].sum()

In [ ]:
df_eda = df.copy()

df_eda["Machine Type"] = "H"
df_eda.loc[df_eda["Type_L"] == 1, "Machine Type"] = "L"
df_eda.loc[df_eda["Type_M"] == 1, "Machine Type"] = "M"

In [ ]:
sns.countplot(data=df_eda, x="Machine Type")

plt.title("Distribution of Machine Types")
plt.show()

### 6.3 Distribution of Numerical Features

In [ ]:
numerical_features = [
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min"
]

In [ ]:
for feature in numerical_features:
    plt.figure(figsize=(8,4))

    sns.histplot(df[feature], kde=True)

    plt.title(feature)

    plt.show()

### 6.4 Outliers

In [ ]:
for feature in numerical_features:

    plt.figure(figsize=(8,4))

    sns.boxplot(x=df[feature])

    plt.title(feature)

    plt.show()

### 6.5 Correlation

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()

### 6.6 Features of Failed and Non-Failed Machines

In [ ]:
features = [
    "Air_temperature_K",
    "Process_temperature_K",
    "Rotational_speed_rpm",
    "Torque_Nm",
    "Tool_wear_min"
]

for feature in features:
    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="Machine_failure",
        y=feature
    )

    plt.title(f"{feature} by Machine Failure")
    plt.tight_layout()
    plt.show()

### 6.7 Summary of Exploratory Data Analysis

The exploratory data analysis identified several important characteristics of the dataset. Machine failures are relatively rare, resulting in an imbalanced classification problem. Additionally, the machine types are not evenly distributed, with Type L representing the largest proportion of observations.

The numerical variables exhibit different distributions. Air temperature, process temperature, and torque are approximately normally distributed, while rotational speed is positively skewed. The boxplots show that machines experiencing failures generally operate at higher air and process temperatures, higher torque, greater tool wear, and lower rotational speeds than machines without failures.

The correlation matrix reveals strong relationships between air temperature and process temperature (0.88) and between rotational speed and torque (-0.88). However, the target variable shows only weak linear correlations with the individual features. Together with the boxplots, this suggests that machine failures are influenced by combinations of operating conditions rather than a single variable, making the dataset well suited for machine learning models that can learn complex, non-linear relationships.

## 7. Machine Learning Preparation

### 7.1 Load Processed Dataset

In [ ]:
df = pd.read_csv("data/processed/ai4i2020_processed.csv")

### 7.2 Feature / Target Split

In [ ]:
X = df.drop(columns=["Machine_failure"])
y = df["Machine_failure"]

### 7.3 Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)

y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

print("Train/test split saved.")

## 8. Unsupervised Learning

Unlike supervised learning, unsupervised learning does not use a target variable. Instead, it is used to discover hidden patterns and relationships within the data. In this project, K-Means clustering is used to identify groups of machines with similar operating characteristics, while PCA and UMAP are used to visualize these clusters in two dimensions.


### 8.1 K-means Clustering

In [ ]:
# Load the training data
X_train = pd.read_csv("data/processed/X_train.csv")

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Calculate inertia for k = 1 to 10
inertia = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_train_scaled)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(range(1,11), inertia, marker="o")

plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")

plt.title("Elbow Method for Optimal Number of Clusters")

plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_train_scaled)

In [ ]:
X_train_clusters = X_train.copy()

X_train_clusters["Cluster"] = clusters

In [ ]:
X_train_clusters.groupby("Cluster").mean()

In [ ]:
X_train_clusters["Machine failure"] = y_train.values

pd.crosstab(
    X_train_clusters["Cluster"],
    X_train_clusters["Machine failure"]
)

In [ ]:
cluster_summary = X_train_clusters.groupby("Cluster")["Machine failure"].agg(
    Total="count",
    Failures="sum"
)

cluster_summary["Failure Rate (%)"] = (
    cluster_summary["Failures"] / cluster_summary["Total"] * 100
).round(2)

cluster_summary

In [ ]:
from sklearn.metrics import silhouette_score
sil_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train_scaled)
    sil_scores.append(silhouette_score(X_train_scaled, km.labels_))

In [ ]:
from sklearn.metrics import silhouette_score

inertia = []
sil_scores = []

k_range = range(2, 11)  # silhouette needs k >= 2

for k in k_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_train_scaled)

    inertia.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X_train_scaled, labels))

fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = "tab:blue"
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia", color=color1)
ax1.plot(list(k_range), inertia, marker="o", color=color1, label="Inertia")
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
color2 = "tab:red"
ax2.set_ylabel("Silhouette Score", color=color2)
ax2.plot(list(k_range), sil_scores, marker="s", color=color2, label="Silhouette Score")
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("Elbow Method (Inertia) vs. Silhouette Score")
fig.tight_layout()
plt.show()

for k, sil in zip(k_range, sil_scores):
    print(f"k={k}: silhouette={sil:.4f}")

### 8.2 PCA
Principal Component Analysis (PCA) is a dimensionality reduction technique that transforms high-dimensional data into a smaller number of components while preserving as much of the original variation as possible. In this project, PCA is used to visualize the clusters identified by the K-Means algorithm in two dimensions.

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_train_scaled)

In [ ]:
pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = clusters

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="viridis",
    alpha=0.7
)

plt.title("PCA Projection of K-Means Clusters")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.legend(title="Cluster")

plt.show()

In [ ]:
print(
    f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}"
)

PCA reduced the seven-dimensional feature space to two principal components while retaining **53.6%** of the total variance. This made it possible to visualize the K-Means clusters in two dimensions.

The projection shows moderate separation between the clusters, with Cluster 2 appearing more distinct than Clusters 0 and 1, which exhibit greater overlap. This suggests that while the first two principal components capture much of the dataset's structure, some information is inevitably lost during dimensionality reduction. Overall, the PCA visualization supports the presence of meaningful underlying patterns in the data.

### 8.3 UMAP

In [ ]:
reducer = umap.UMAP(random_state=42)

X_umap = reducer.fit_transform(X_train_scaled)

In [ ]:
umap_df = pd.DataFrame(
    X_umap,
    columns=["UMAP1", "UMAP2"]
)

umap_df["Cluster"] = clusters

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(
    data=umap_df,
    x="UMAP1",
    y="UMAP2",
    hue="Cluster",
    palette="viridis",
    alpha=0.7
)

plt.title("UMAP Projection of K-Means Clusters")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")

plt.legend(title="Cluster")

plt.show()

UMAP provided a clearer visualization of the K-Means clusters than PCA by preserving the local structure of the data. Cluster 0 is almost completely separated from the remaining clusters, while Clusters 1 and 2 exhibit less overlap than in the PCA projection.

The improved separation suggests that the dataset contains meaningful non-linear relationships that are not fully captured by PCA. These results further support the presence of distinct operating profiles within the data and demonstrate the value of modern dimensionality reduction techniques for exploring complex industrial datasets.

### 8.4 Summary

The optimal number of clusters was evaluated using both the Elbow Method and the Silhouette Score. Although the silhouette score was highest for **k = 2**, a three-cluster solution was selected because it provided more interpretable operating profiles and revealed a distinct group of machines with a substantially higher failure rate.

K-Means successfully identified three operating profiles without using the machine failure labels. Cluster 2 exhibited a failure rate of **5.15%**, compared with **2.44%** for Clusters 0 and 1. PCA and UMAP both visualized these clusters, with UMAP providing clearer separation by preserving non-linear relationships in the data. Together, these results demonstrate that the dataset contains meaningful underlying structure and provide a strong foundation for the supervised machine learning models developed in the next section.

## 9. Supervised Learning

Unlike the previous section, supervised learning uses historical machine failure labels to train models that can predict whether a machine is likely to fail. Several algorithms are evaluated and compared to determine which model provides the best predictive performance for industrial maintenance applications.

### 9.1 Logistic Regression (Baseline Model)
Logistic Regression is used as a baseline classification model. Although it assumes a linear relationship between the input features and the target variable, it provides a useful benchmark against which more complex machine learning models can be compared.

In [ ]:
# Load the saved train/test split
X_train = pd.read_csv("data/processed/X_train.csv")
X_test = pd.read_csv("data/processed/X_test.csv")

y_train = pd.read_csv("data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("data/processed/y_test.csv").squeeze()

# Scale the features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_reg_path = MODEL_DIR / "logistic_regression.pkl"

if log_reg_path.exists():
    print("Loading Logistic Regression model...")
    log_reg = joblib.load(log_reg_path)
else:
    print("Training Logistic Regression model...")

    log_reg = LogisticRegression(random_state=42)

    log_reg.fit(X_train_scaled, y_train)

    joblib.dump(log_reg, log_reg_path)

In [ ]:
y_pred_lr = log_reg.predict(X_test_scaled)

y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

### 9.2 Random Forest
Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve predictive performance and reduce overfitting. Unlike Logistic Regression, Random Forest can capture complex, non-linear relationships between the predictor variables and the target variable.

In [ ]:
rf_path = MODEL_DIR / "random_forest.pkl"

if rf_path.exists():
    print("Loading Random Forest model...")
    rf = joblib.load(rf_path)
else:
    print("Training Random Forest model...")

    rf = RandomForestClassifier(
        random_state=42
    )

    rf.fit(X_train, y_train)

    joblib.dump(rf, rf_path)

In [ ]:
y_pred_rf = rf.predict(X_test)

y_prob_rf = rf.predict_proba(X_test)[:, 1]

### 9.3 XGBoost
Extreme Gradient Boosting (XGBoost) is an advanced ensemble learning algorithm that builds decision trees sequentially, with each tree improving upon the errors of the previous one. It is widely used in industrial machine learning applications because of its high predictive performance and ability to model complex, non-linear relationships.

In [ ]:
xgb_path = MODEL_DIR / "xgboost.pkl"

if xgb_path.exists():
    print("Loading XGBoost model...")
    xgb = joblib.load(xgb_path)
else:
    print("Training XGBoost model...")

    xgb = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        eval_metric="logloss"
    )

    xgb.fit(X_train, y_train)

    joblib.dump(xgb, xgb_path)

### 9.4 Artificial Neural Network (ANN)

Artificial Neural Networks (ANNs) are deep learning models capable of learning complex, non-linear relationships between input features and the target variable. In this project, a feedforward neural network is trained to predict machine failures and compared with the traditional machine learning models.

In [ ]:
print(type(X_train))

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

y_train = y_train.to_numpy(dtype=np.float32)
y_test = y_test.to_numpy(dtype=np.float32)

print(type(X_train))
print(X_train.dtype)

In [ ]:
# ============================
# Artificial Neural Network
# ============================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Load processed dataset
# ----------------------------

df = pd.read_csv("data/processed/ai4i2020_processed.csv")

# ----------------------------
# Feature / Target Split
# ----------------------------

X = df.drop("Machine_failure", axis=1)
y = df["Machine_failure"]

# ----------------------------
# Train / Test Split
# ----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ----------------------------
# Feature Scaling
# ----------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

y_train = y_train.to_numpy(dtype=np.float32)
y_test = y_test.to_numpy(dtype=np.float32)

# ----------------------------
# Sanity Checks
# ----------------------------

print(type(X_train))
print(type(y_train))
print(X_train.shape)
print(y_train.shape)
print(X_train.dtype)
print(y_train.dtype)

# ----------------------------
# Build ANN
# ----------------------------

model = tf.keras.Sequential([
    tf.keras.Input(shape=(7,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

# ----------------------------
# Compile
# ----------------------------

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# ----------------------------
# Train
# ----------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# ----------------------------
# Save Model
# ----------------------------

model.save("models/ann.keras")

print("ANN model saved successfully.")

## 10. Model Evaluation

### 10.1. Logistic Regression Evaluation
Confusion Matrix
Accuracy
Precision
Recall
F1-score
ROC Curve
ROC-AUC

### 10.2 Random Forest Evaluation
Confusion Matrix
Accuracy
Precision
Recall
F1-score
ROC Curve
ROC-AUC
Feature Importance

### 10.3 XGBoost Evaluation
Confusion Matrix
Accuracy
Precision
Recall
F1-score
ROC Curve
ROC-AUC
Feature Importance

### 10.4 Artificial Neural Network (ANN) Evaluation
Confusion Matrix
Accuracy
Precision
Recall
F1-score
ROC Curve
ROC-AUC
Feature Importance

### 10.5 Model Comparison
Accuracy
Percision
Recall
F1
ROC-AUC
Training Time

## 11. AI Decision Support System

The machine learning model is used as part of a decision-support system rather
than only producing a binary failure prediction.

The XGBoost model estimates the probability that a machine will experience a
failure. This probability is then converted into a risk level using predefined
risk thresholds.

Based on the risk level, the system generates a maintenance recommendation.

The decision-support workflow is:

1. Machine sensor data is received.
2. The XGBoost model predicts failure probability.
3. The probability is converted into a risk level.
4. A maintenance recommendation is generated.
5. The result is presented to the user through the Streamlit application.

This allows the model output to be translated into information that can support
maintenance decisions.

The system therefore combines machine learning prediction with a simple
decision-support layer.

Sensor Data<br>
↓<br>
XGBoost<br>
↓<br>
Failure Probability<br>
↓<br>
Risk Level<br>
↓<br>
Maintenance Recommendation<br>
↓<br>
Decision Support Dashboard

## 12. Streamlit Dashboard (Deployment)

The trained machine learning models and prediction logic were integrated into an
interactive web application using Streamlit.

The application provides several components:

- Dashboard with an overview of machine risk
- Interactive machine failure prediction
- Model analytics and visualizations
- Project and methodology information
- Live machine monitoring

The prediction page allows users to enter machine operating conditions and
receive a failure probability, risk level, and maintenance recommendation.

The Live Monitoring page extends the application with a simulated real-time
data pipeline using Apache Kafka. Sensor readings are sent by a Kafka producer,
processed by a consumer, and passed through the trained XGBoost model.

The resulting predictions are stored and displayed in the Streamlit interface,
where maintenance alerts can be reviewed and managed.

The application therefore demonstrates the complete workflow from incoming
machine data to machine learning prediction and maintenance decision support.

## 13. Live Monitoring

The project also includes a simulated real-time predictive maintenance monitoring system.

Sensor observations are streamed through Apache Kafka. A Kafka consumer receives the incoming data and uses the trained XGBoost model to generate failure probabilities. These probabilities are then classified into risk levels and used to generate maintenance recommendations.

The predictions and maintenance alerts are stored and displayed in the Streamlit Live Monitoring page.

The Live Monitoring interface provides:

- Real-time machine predictions
- Failure probability and risk level
- Active maintenance alerts
- Sensor readings at the time an alert was detected
- Maintenance recommendations
- Maintenance notes and incident status
- Start/Pause controls for the demonstration


### Live Monitoring Architecture

Sensor Observations<br>
↓<br>
Apache Kafka<br>
↓<br>
Kafka Consumer<br>
↓<br>
XGBoost Model<br>
↓<br>
Failure Probability<br>
↓<br>
Risk Classification<br>
↓<br>
Maintenance Recommendation<br>
↓<br>
Predictions / Maintenance Alerts<br>
↓<br>
Streamlit Live Monitoring

## 14. Maintenance Recommendations

The system generates maintenance recommendations based on the predicted
failure probability and assigned risk level.

Recommendations are intended to support maintenance decisions rather than
replace human decision-making.

The recommendation system considers:

- Machine type
- Failure probability
- Risk level

Higher-risk predictions trigger more urgent maintenance recommendations,
while low-risk predictions require no immediate intervention.

## 15. Conclusion & Future Work

### Conclusion

This project demonstrates an end-to-end AI-powered predictive maintenance
system, from data preprocessing and exploratory analysis to machine learning,
prediction, and deployment.

Several supervised learning models were evaluated, including Logistic
Regression, Random Forest, XGBoost, and an Artificial Neural Network.
XGBoost provided the strongest overall performance and was selected for the
predictive maintenance system.

The final application provides:

- Machine failure probability predictions
- Risk classification
- Maintenance recommendations
- Interactive data visualization
- Real-time machine monitoring using Apache Kafka
- Automated maintenance alerts
- A Streamlit dashboard for decision support

The live monitoring component extends the project from an offline machine
learning workflow into a simulated real-time predictive maintenance system.
Sensor observations are streamed through Kafka, processed by the consumer,
evaluated by the XGBoost model, and displayed in the Streamlit application.

### Lessons Learned

The project provided practical experience with:

- Data preprocessing and feature engineering
- Supervised and unsupervised machine learning
- Model evaluation and comparison
- Feature importance
- Model deployment with Streamlit
- Real-time data streaming with Apache Kafka
- Integrating machine learning predictions into a decision-support system

### Future Work

Possible future improvements include:

- Remaining Useful Life (RUL) prediction
- Improved real-time sensor integration
- More sophisticated maintenance scheduling
- Maintenance cost estimation
- Explainable AI using SHAP
- Database storage for production-scale monitoring
- Cloud deployment and API integration
- More advanced anomaly detection